In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import os

# ============================================================
# INPUT FILES
# ============================================================

FILES = {
    "Loop": "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet",
    "PEDAP": "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAPdropped.parquet",
    "ReplaceBG": "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBGdropped.parquet"
}

# New output file
OUTPUT = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_merged_with_source.parquet"


# ============================================================
# REMOVE OLD OUTPUT IF IT EXISTS
# ============================================================

if os.path.exists(OUTPUT):
    os.remove(OUTPUT)
    print("Removed existing output file.")


# ============================================================
# CHECK SCHEMAS
# ============================================================

print("=" * 100)
print("CHECKING INPUT SCHEMAS")
print("=" * 100)

schemas = {}

for source, path in FILES.items():

    pf = pq.ParquetFile(path)
    schemas[source] = pf.schema_arrow

    print(f"\n{source}")
    print(f"Rows: {pf.metadata.num_rows:,}")
    print(f"Row groups: {pf.num_row_groups:,}")
    print(f"Columns: {len(pf.schema_arrow.names)}")


# Make sure all original schemas match
base_schema = schemas["Loop"]

for source, schema in schemas.items():

    if schema != base_schema:
        raise ValueError(
            f"Schema mismatch between Loop and {source}"
        )

print("\n✅ All input schemas match.")


# ============================================================
# CREATE OUTPUT SCHEMA
# ============================================================

output_schema = base_schema.append(
    pa.field("source", pa.string())
)

print("\nOutput columns:")
print(output_schema.names)


# ============================================================
# STREAM MERGE
# ============================================================

writer = None

total_rows = 0

for source, path in FILES.items():

    print("\n" + "=" * 100)
    print(f"ADDING: {source}")
    print("=" * 100)

    pf = pq.ParquetFile(path)

    source_rows = 0

    for rg in range(pf.num_row_groups):

        # Read one row group only
        table = pf.read_row_group(rg)

        # Add source column
        source_array = pa.array(
            [source] * table.num_rows,
            type=pa.string()
        )

        table = table.append_column(
            "source",
            source_array
        )

        # Create writer using final schema
        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT,
                output_schema,
                compression="snappy"
            )

        writer.write_table(table)

        source_rows += table.num_rows
        total_rows += table.num_rows

        if (rg + 1) % 100 == 0 or rg == pf.num_row_groups - 1:

            print(
                f"  Row groups: "
                f"{rg + 1}/{pf.num_row_groups} | "
                f"Rows: {source_rows:,}"
            )


    print(
        f"✅ {source} complete: "
        f"{source_rows:,} rows"
    )


# Close writer
if writer is not None:
    writer.close()


# ============================================================
# VERIFY OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("VERIFYING OUTPUT")
print("=" * 100)

out_pf = pq.ParquetFile(OUTPUT)

print(f"\nOutput file:")
print(OUTPUT)

print(f"\nRows: {out_pf.metadata.num_rows:,}")
print(f"Row groups: {out_pf.num_row_groups:,}")
print(f"Columns: {len(out_pf.schema_arrow.names)}")

print("\nColumns:")
for col in out_pf.schema_arrow.names:
    print(f"  - {col}")


# ============================================================
# VERIFY SOURCE COUNTS
# ============================================================

print("\n" + "=" * 100)
print("VERIFYING SOURCE COUNTS")
print("=" * 100)

source_counts = {}

for rg in range(out_pf.num_row_groups):

    table = out_pf.read_row_group(
        rg,
        columns=["source"]
    )

    sources = table["source"].to_pylist()

    for source in sources:
        source_counts[source] = (
            source_counts.get(source, 0) + 1
        )


for source, count in source_counts.items():

    print(
        f"{source:12s}: "
        f"{count:,} rows"
    )


print("\n" + "=" * 100)
print("FINAL CHECK")
print("=" * 100)

print(
    f"\nExpected total: "
    f"{sum(pq.ParquetFile(path).metadata.num_rows for path in FILES.values()):,}"
)

print(
    f"Actual total:   "
    f"{out_pf.metadata.num_rows:,}"
)

if (
    out_pf.metadata.num_rows ==
    sum(
        pq.ParquetFile(path).metadata.num_rows
        for path in FILES.values()
    )
):
    print("\n✅ MERGE SUCCESSFUL")
    print("✅ Source column added")
    print("✅ Row count matches input files")

else:
    print("\n❌ ROW COUNT MISMATCH")

CHECKING INPUT SCHEMAS


FileNotFoundError: [Errno 2] Failed to open local file '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet'. Detail: [errno 2] No such file or directory

# Step 1 — CGM level → bolus relationship

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

PATH = '/Volumes/HIKSEMI/grad sakr4/datasets/metabonet_merged_with_source.parquet'

pf = pq.ParquetFile(PATH)

# ============================================================
# CGM BINS
# ============================================================

bins = [
    0,
    54,
    70,
    80,
    100,
    120,
    140,
    160,
    180,
    200,
    250,
    300,
    400,
    1000
]

labels = [
    "<54",
    "54-69",
    "70-79",
    "80-99",
    "100-119",
    "120-139",
    "140-159",
    "160-179",
    "180-199",
    "200-249",
    "250-299",
    "300-399",
    "400+"
]

# ============================================================
# STORAGE
# ============================================================

stats = {
    label: {
        "rows": 0,
        "bolus_known": 0,
        "bolus_positive": 0,
        "bolus_sum": 0.0,
        "bolus_values": []
    }
    for label in labels
}

# ============================================================
# STREAM THROUGH DATASET
# ============================================================

for rg in range(pf.num_row_groups):

    table = pf.read_row_group(
        rg,
        columns=["CGM", "bolus"]
    )

    df = table.to_pandas()

    df["CGM"] = pd.to_numeric(
        df["CGM"],
        errors="coerce"
    )

    df["bolus"] = pd.to_numeric(
        df["bolus"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["CGM"]
    )

    # Keep reasonable CGM values
    df = df[
        (df["CGM"] >= 0) &
        (df["CGM"] <= 1000)
    ]

    df["cgm_bin"] = pd.cut(
        df["CGM"],
        bins=bins,
        labels=labels,
        right=False
    )

    for label, group in df.groupby(
        "cgm_bin",
        observed=True
    ):

        s = stats[label]

        s["rows"] += len(group)

        known = group["bolus"].notna()

        s["bolus_known"] += known.sum()

        positive = group.loc[
            known,
            "bolus"
        ] > 0

        s["bolus_positive"] += positive.sum()

        positive_values = group.loc[
            known & (group["bolus"] > 0),
            "bolus"
        ]

        s["bolus_sum"] += positive_values.sum()

        # Keep values for percentiles
        s["bolus_values"].extend(
            positive_values.tolist()
        )

    if (rg + 1) % 100 == 0:
        print(
            f"Processed "
            f"{rg + 1}/{pf.num_row_groups} row groups"
        )


# ============================================================
# CREATE RESULTS TABLE
# ============================================================

results = []

for label in labels:

    s = stats[label]

    if s["bolus_known"] > 0:

        bolus_probability = (
            s["bolus_positive"] /
            s["bolus_known"] *
            100
        )

    else:
        bolus_probability = np.nan

    if len(s["bolus_values"]) > 0:

        values = np.array(
            s["bolus_values"]
        )

        median_bolus = np.median(values)
        mean_bolus = np.mean(values)
        p90_bolus = np.percentile(values, 90)
        max_bolus = np.max(values)

    else:

        median_bolus = np.nan
        mean_bolus = np.nan
        p90_bolus = np.nan
        max_bolus = np.nan

    results.append({
        "CGM_range": label,
        "rows": s["rows"],
        "bolus_known": s["bolus_known"],
        "bolus_positive": s["bolus_positive"],
        "bolus_probability_%": bolus_probability,
        "mean_positive_bolus": mean_bolus,
        "median_positive_bolus": median_bolus,
        "P90_positive_bolus": p90_bolus,
        "max_positive_bolus": max_bolus
    })


results_df = pd.DataFrame(results)


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 120)
print("CGM LEVEL → BOLUS RELATIONSHIP")
print("=" * 120)

print(
    results_df.to_string(
        index=False,
        formatters={
            "bolus_probability_%": "{:.3f}".format,
            "mean_positive_bolus": "{:.3f}".format,
            "median_positive_bolus": "{:.3f}".format,
            "P90_positive_bolus": "{:.3f}".format,
            "max_positive_bolus": "{:.3f}".format
        }
    )
)

# Save
OUTPUT = '/Volumes/HIKSEMI/grad sakr4/datasets/cgm_bolus_relationship.csv'

results_df.to_csv(
    OUTPUT,
    index=False
)

print(
    f"\nSaved to:\n{OUTPUT}"
)

Processed 100/1214 row groups
Processed 200/1214 row groups
Processed 300/1214 row groups
Processed 400/1214 row groups
Processed 500/1214 row groups
Processed 600/1214 row groups
Processed 700/1214 row groups
Processed 800/1214 row groups
Processed 900/1214 row groups
Processed 1000/1214 row groups
Processed 1100/1214 row groups
Processed 1200/1214 row groups

CGM LEVEL → BOLUS RELATIONSHIP
CGM_range     rows  bolus_known  bolus_positive bolus_probability_% mean_positive_bolus median_positive_bolus P90_positive_bolus max_positive_bolus
      <54   695063       625768            7419               1.186               1.709                 0.700              4.760             24.000
    54-69  3013225      2736500           44400               1.623               1.992                 1.000              5.000             33.400
    70-79  4459934      4079566           81457               1.997               2.127                 1.300              5.150             43.400
    80-99 150

In [ ]:
import os
import pyarrow.parquet as pq

pf = pq.ParquetFile(PATH)

print("File size:",
      round(os.path.getsize(PATH) / 1024**3, 2),
      "GB")

print("Rows:",
      pf.metadata.num_rows)

print("Row groups:",
      pf.num_row_groups)

print("Average rows/row group:",
      round(pf.metadata.num_rows / pf.num_row_groups))

File size: 1.31 GB
Rows: 120994310
Row groups: 1214
Average rows/row group: 99666


In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

pf = pq.ParquetFile(PATH)

carbs_positive = 0
carbs_zero = 0
bolus_positive = 0

for rg in range(pf.num_row_groups):

    table = pf.read_row_group(
        rg,
        columns=["carbs", "bolus"]
    )

    df = table.to_pandas()

    carbs = pd.to_numeric(df["carbs"], errors="coerce")
    bolus = pd.to_numeric(df["bolus"], errors="coerce")

    carbs_positive += ((carbs > 0) & np.isfinite(carbs)).sum()
    carbs_zero += ((carbs == 0) & np.isfinite(carbs)).sum()
    bolus_positive += ((bolus > 0) & np.isfinite(bolus)).sum()

    if (rg + 1) % 100 == 0:
        print(f"{rg + 1}/{pf.num_row_groups}")

print("\n==============================")
print(f"Positive carb events : {carbs_positive:,}")
print(f"Zero-carb rows       : {carbs_zero:,}")
print(f"Positive bolus events: {bolus_positive:,}")
print("==============================")

100/1214
200/1214
300/1214
400/1214
500/1214
600/1214
700/1214
800/1214
900/1214
1000/1214
1100/1214
1200/1214

Positive carb events : 1,642,370
Zero-carb rows       : 101,353,163
Positive bolus events: 3,499,811


In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import time
import os

# ============================================================
# CONFIGURATION
# ============================================================

PATH = "/Volumes/HIKSEMI/grad sakr4/datasets/metabonet_merged_with_source.parquet"

OUTPUT = "/Volumes/HIKSEMI/grad sakr4/datasets/cgm_bolus_relationship.csv"

WINDOWS = np.array([5, 10, 15, 30, 60], dtype=np.int64)

# ============================================================
# OPEN PARQUET
# ============================================================

pf = pq.ParquetFile(PATH)

print("=" * 80)
print("OPTIMIZED CARBS → BOLUS ANALYSIS")
print("=" * 80)

print(f"File: {PATH}")
print(f"Rows: {pf.metadata.num_rows:,}")
print(f"Row groups: {pf.num_row_groups:,}")
print()

# ============================================================
# STATISTICS
# ============================================================

stats = {
    w: {
        "carb_events": 0,
        "carb_events_followed_by_bolus": 0,
        "no_carb_windows": 0,
        "no_carb_followed_by_bolus": 0
    }
    for w in WINDOWS
}

# ============================================================
# PATIENT BUFFER
# ============================================================

patient_buffer = {}

start_time = time.time()

# ============================================================
# PROCESS ROW GROUPS
# ============================================================

for rg in range(pf.num_row_groups):

    table = pf.read_row_group(
        rg,
        columns=["id", "date", "carbs", "bolus"]
    )

    df = table.to_pandas()

    # --------------------------------------------------------
    # Clean data
    # --------------------------------------------------------

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df["carbs"] = pd.to_numeric(
        df["carbs"],
        errors="coerce"
    )

    df["bolus"] = pd.to_numeric(
        df["bolus"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["id", "date"]
    )

    # --------------------------------------------------------
    # Process patients
    # --------------------------------------------------------

    for patient_id, g in df.groupby(
        "id",
        sort=False
    ):

        # ----------------------------------------------------
        # Restore previous 60-minute buffer
        # ----------------------------------------------------

        if patient_id in patient_buffer:

            g = pd.concat(
                [
                    patient_buffer[patient_id],
                    g
                ],
                ignore_index=True
            )

        # ----------------------------------------------------
        # Sort patient chronologically
        # ----------------------------------------------------

        g = g.sort_values(
            "date"
        ).reset_index(drop=True)

        # ----------------------------------------------------
        # Extract arrays
        # ----------------------------------------------------

        dates = g["date"].values.astype(
            "datetime64[ns]"
        )

        carbs = g["carbs"].to_numpy()

        bolus = g["bolus"].to_numpy()

        # ----------------------------------------------------
        # Positive bolus timestamps
        # ----------------------------------------------------

        bolus_mask = (
            np.isfinite(bolus) &
            (bolus > 0)
        )

        bolus_times = dates[bolus_mask]

        # IMPORTANT:
        # dates are already sorted.
        # Therefore bolus_times is also sorted.
        # ----------------------------------------------------

        # ====================================================
        # CARB EVENTS
        # ====================================================

        carb_mask = (
            np.isfinite(carbs) &
            (carbs > 0)
        )

        carb_times = dates[carb_mask]

        if len(carb_times) > 0:

            # ------------------------------------------------
            # For each carb event, find the FIRST bolus
            # strictly AFTER the carb timestamp.
            #
            # searchsorted replaces:
            #
            # for j in bolus_indices:
            #     ...
            # ------------------------------------------------

            first_bolus_idx = np.searchsorted(
                bolus_times,
                carb_times,
                side="right"
            )

            # ------------------------------------------------
            # Analyze each window
            # ------------------------------------------------

            for w in WINDOWS:

                stats[w]["carb_events"] += len(
                    carb_times
                )

                window_end = (
                    carb_times +
                    np.timedelta64(int(w), "m")
                )

                valid = (
                    first_bolus_idx <
                    len(bolus_times)
                )

                # Avoid indexing outside bolus_times
                safe_idx = np.minimum(
                    first_bolus_idx,
                    max(len(bolus_times) - 1, 0)
                )

                if len(bolus_times) > 0:

                    first_bolus_times = (
                        bolus_times[safe_idx]
                    )

                    followed = (
                        valid &
                        (first_bolus_times <= window_end)
                    )

                    stats[w][
                        "carb_events_followed_by_bolus"
                    ] += int(
                        followed.sum()
                    )

        # ====================================================
        # ZERO-CARB BASELINE
        # ====================================================

        zero_mask = (
            np.isfinite(carbs) &
            (carbs == 0)
        )

        zero_times = dates[zero_mask]

        if len(zero_times) > 0:

            # ------------------------------------------------
            # Find first bolus after every zero-carb timestamp
            # ------------------------------------------------

            first_bolus_idx = np.searchsorted(
                bolus_times,
                zero_times,
                side="right"
            )

            for w in WINDOWS:

                stats[w]["no_carb_windows"] += len(
                    zero_times
                )

                window_end = (
                    zero_times +
                    np.timedelta64(int(w), "m")
                )

                valid = (
                    first_bolus_idx <
                    len(bolus_times)
                )

                if len(bolus_times) > 0:

                    safe_idx = np.minimum(
                        first_bolus_idx,
                        len(bolus_times) - 1
                    )

                    first_bolus_times = (
                        bolus_times[safe_idx]
                    )

                    followed = (
                        valid &
                        (first_bolus_times <= window_end)
                    )

                    stats[w][
                        "no_carb_followed_by_bolus"
                    ] += int(
                        followed.sum()
                    )

        # ====================================================
        # SAVE 60-MINUTE BUFFER
        # ====================================================

        cutoff = (
            g["date"].max() -
            pd.Timedelta(minutes=60)
        )

        patient_buffer[patient_id] = (
            g[
                g["date"] >= cutoff
            ][
                ["id", "date", "carbs", "bolus"]
            ].copy()
        )

    # ========================================================
    # PROGRESS
    # ========================================================

    if (rg + 1) % 25 == 0 or rg == 0:

        elapsed = time.time() - start_time

        progress = (
            (rg + 1) /
            pf.num_row_groups *
            100
        )

        estimated_total = (
            elapsed /
            (rg + 1) *
            pf.num_row_groups
        )

        remaining = (
            estimated_total -
            elapsed
        )

        print(
            f"Row groups: "
            f"{rg + 1:,}/{pf.num_row_groups:,} "
            f"({progress:.1f}%) | "
            f"Elapsed: {elapsed / 60:.1f} min | "
            f"ETA: {remaining / 60:.1f} min"
        )


# ============================================================
# RESULTS
# ============================================================

results = []

for w in WINDOWS:

    s = stats[w]

    # --------------------------------------------------------
    # Carb probability
    # --------------------------------------------------------

    if s["carb_events"] > 0:

        carb_probability = (
            s["carb_events_followed_by_bolus"] /
            s["carb_events"] *
            100
        )

    else:

        carb_probability = np.nan

    # --------------------------------------------------------
    # Baseline probability
    # --------------------------------------------------------

    if s["no_carb_windows"] > 0:

        baseline_probability = (
            s["no_carb_followed_by_bolus"] /
            s["no_carb_windows"] *
            100
        )

    else:

        baseline_probability = np.nan

    # --------------------------------------------------------
    # Risk difference
    # --------------------------------------------------------

    risk_difference = (
        carb_probability -
        baseline_probability
    )

    # --------------------------------------------------------
    # Relative risk
    # --------------------------------------------------------

    if (
        np.isfinite(baseline_probability) and
        baseline_probability > 0
    ):

        relative_risk = (
            carb_probability /
            baseline_probability
        )

    else:

        relative_risk = np.nan

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    results.append({

        "window_minutes": int(w),

        "carb_events":
            s["carb_events"],

        "carb_followed_by_bolus":
            s["carb_events_followed_by_bolus"],

        "bolus_after_carb_%":
            carb_probability,

        "zero_carb_windows":
            s["no_carb_windows"],

        "bolus_after_zero_carb_%":
            baseline_probability,

        "risk_difference_%":
            risk_difference,

        "relative_risk":
            relative_risk
    })


# ============================================================
# DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

# ============================================================
# DISPLAY
# ============================================================

print("\n")
print("=" * 120)
print("CARBS → SUBSEQUENT BOLUS RELATIONSHIP")
print("=" * 120)

print(
    results_df.to_string(
        index=False,
        formatters={
            "bolus_after_carb_%":
                "{:.3f}".format,

            "bolus_after_zero_carb_%":
                "{:.3f}".format,

            "risk_difference_%":
                "{:.3f}".format,

            "relative_risk":
                "{:.3f}".format
        }
    )
)

# ============================================================
# SAVE
# ============================================================

results_df.to_csv(
    OUTPUT,
    index=False
)

total_time = (
    time.time() -
    start_time
)

print()
print("=" * 80)
print(f"Saved to: {OUTPUT}")
print(
    f"Total runtime: "
    f"{total_time / 60:.2f} minutes"
)
print("=" * 80)

OPTIMIZED CARBS → BOLUS ANALYSIS
File: /Volumes/HIKSEMI/grad sakr4/datasets/metabonet_merged_with_source.parquet
Rows: 120,994,310
Row groups: 1,214

Row groups: 1/1,214 (0.1%) | Elapsed: 0.0 min | ETA: 0.4 min
Row groups: 25/1,214 (2.1%) | Elapsed: 0.0 min | ETA: 0.5 min
Row groups: 50/1,214 (4.1%) | Elapsed: 0.0 min | ETA: 0.4 min
Row groups: 75/1,214 (6.2%) | Elapsed: 0.0 min | ETA: 0.4 min
Row groups: 100/1,214 (8.2%) | Elapsed: 0.0 min | ETA: 0.4 min
Row groups: 125/1,214 (10.3%) | Elapsed: 0.0 min | ETA: 0.4 min
Row groups: 150/1,214 (12.4%) | Elapsed: 0.1 min | ETA: 0.4 min
Row groups: 175/1,214 (14.4%) | Elapsed: 0.1 min | ETA: 0.4 min
Row groups: 200/1,214 (16.5%) | Elapsed: 0.1 min | ETA: 0.4 min
Row groups: 225/1,214 (18.5%) | Elapsed: 0.1 min | ETA: 0.4 min
Row groups: 250/1,214 (20.6%) | Elapsed: 0.1 min | ETA: 0.4 min
Row groups: 275/1,214 (22.7%) | Elapsed: 0.1 min | ETA: 0.4 min
Row groups: 300/1,214 (24.7%) | Elapsed: 0.1 min | ETA: 0.3 min
Row groups: 325/1,214 (26.8%

In [3]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

PATH = '/Users/mariammohamed/Documents/uni/sem8/grad/grad sakr4/datasets/metabonet_merged_with_source.parquet'


pf = pq.ParquetFile(PATH)

WINDOWS = [5, 10, 15, 30, 60]

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

stats = {
    w: {
        "carb_events": 0,
        "carb_events_followed_by_bolus": 0,
        "no_carb_windows": 0,
        "no_carb_followed_by_bolus": 0
    }
    for w in WINDOWS
}

# We process one patient at a time within each row group.
# Keep recent events across row-group boundaries.

patient_buffer = {}

# ------------------------------------------------------------
# Process row groups
# ------------------------------------------------------------

for rg in range(pf.num_row_groups):

    table = pf.read_row_group(
        rg,
        columns=["id", "date", "carbs", "bolus"]
    )

    df = table.to_pandas()

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df["carbs"] = pd.to_numeric(
        df["carbs"],
        errors="coerce"
    )

    df["bolus"] = pd.to_numeric(
        df["bolus"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["id", "date"]
    )

    # --------------------------------------------------------
    # Process each patient
    # --------------------------------------------------------

    for patient_id, g in df.groupby(
        "id",
        sort=False
    ):

        # Combine previous buffered records with current group
        if patient_id in patient_buffer:

            g = pd.concat(
                [
                    patient_buffer[patient_id],
                    g
                ],
                ignore_index=True
            )

        # Sort chronologically for THIS patient
        g = g.sort_values(
            "date"
        ).reset_index(drop=True)

        dates = g["date"].values
        carbs = g["carbs"].values
        bolus = g["bolus"].values

        n = len(g)

        # ----------------------------------------------------
        # Analyze positive carb events
        # ----------------------------------------------------

        carb_indices = np.where(
            np.isfinite(carbs) &
            (carbs > 0)
        )[0]

        bolus_indices = np.where(
            np.isfinite(bolus) &
            (bolus > 0)
        )[0]

        for i in carb_indices:

            carb_time = dates[i]

            for w in WINDOWS:

                stats[w]["carb_events"] += 1

                window_end = (
                    carb_time +
                    np.timedelta64(w, "m")
                )

                # Is there a bolus after this carb event?
                found = False

                for j in bolus_indices:

                    if j <= i:
                        continue

                    if dates[j] <= window_end:

                        found = True
                        break

                    if dates[j] > window_end:
                        break

                if found:

                    stats[w][
                        "carb_events_followed_by_bolus"
                    ] += 1

        # ----------------------------------------------------
        # Create baseline windows
        #
        # Sample timestamps with no positive carbs.
        # We use rows where carbs are explicitly 0.
        # ----------------------------------------------------

        zero_carb_indices = np.where(
            np.isfinite(carbs) &
            (carbs == 0)
        )[0]

        for i in zero_carb_indices:

            current_time = dates[i]

            for w in WINDOWS:

                stats[w]["no_carb_windows"] += 1

                window_end = (
                    current_time +
                    np.timedelta64(w, "m")
                )

                found = False

                for j in bolus_indices:

                    if j <= i:
                        continue

                    if dates[j] <= window_end:

                        found = True
                        break

                    if dates[j] > window_end:
                        break

                if found:

                    stats[w][
                        "no_carb_followed_by_bolus"
                    ] += 1

        # ----------------------------------------------------
        # Keep only recent records.
        #
        # 60 minutes is the largest window, so anything older
        # than 60 minutes cannot affect future calculations.
        # ----------------------------------------------------

        cutoff = (
            g["date"].max() -
            pd.Timedelta(minutes=60)
        )

        patient_buffer[patient_id] = g[
            g["date"] >= cutoff
        ][
            ["id", "date", "carbs", "bolus"]
        ].copy()

    if (rg + 1) % 100 == 0:

        print(
            f"Processed "
            f"{rg + 1}/{pf.num_row_groups}"
        )


# ============================================================
# RESULTS
# ============================================================

results = []

for w in WINDOWS:

    s = stats[w]

    carb_probability = (
        s["carb_events_followed_by_bolus"] /
        s["carb_events"] *
        100
    )

    baseline_probability = (
        s["no_carb_followed_by_bolus"] /
        s["no_carb_windows"] *
        100
    )

    results.append({
        "window_minutes": w,
        "carb_events": s["carb_events"],
        "carb_followed_by_bolus": (
            s["carb_events_followed_by_bolus"]
        ),
        "bolus_after_carb_%": carb_probability,
        "zero_carb_windows": s["no_carb_windows"],
        "bolus_after_zero_carb_%": baseline_probability,
        "risk_difference_%": (
            carb_probability -
            baseline_probability
        ),
        "relative_risk": (
            carb_probability /
            baseline_probability
            if baseline_probability > 0
            else np.nan
        )
    })


results_df = pd.DataFrame(results)

print("\n" + "=" * 120)
print("CARBS → SUBSEQUENT BOLUS RELATIONSHIP")
print("=" * 120)

print(
    results_df.to_string(
        index=False,
        formatters={
            "bolus_after_carb_%": "{:.3f}".format,
            "bolus_after_zero_carb_%": "{:.3f}".format,
            "risk_difference_%": "{:.3f}".format,
            "relative_risk": "{:.3f}".format
        }
    )
)

OUTPUT = (
   '/Users/mariammohamed/Documents/uni/sem8/grad/grad sakr4/datasets/cgm_bolus_relationship.csv'
)

results_df.to_csv(
    OUTPUT,
    index=False
)

print(f"\nSaved to:\n{OUTPUT}")

Processed 100/1214
Processed 200/1214
Processed 300/1214
Processed 400/1214
Processed 500/1214
Processed 600/1214
Processed 700/1214
Processed 800/1214
Processed 900/1214
Processed 1000/1214
Processed 1100/1214
Processed 1200/1214

CARBS → SUBSEQUENT BOLUS RELATIONSHIP
 window_minutes  carb_events  carb_followed_by_bolus bolus_after_carb_%  zero_carb_windows bolus_after_zero_carb_% risk_difference_% relative_risk
              5      1642632                  170356             10.371          101364834                   2.992             7.379         3.467
             10      1642632                  230587             14.038          101364834                   5.307             8.731         2.645
             15      1642632                  285076             17.355          101364834                   7.502             9.853         2.313
             30      1642632                  428144             26.065          101364834                  13.531            12.534         1